In [43]:
import pandas as pd
import os
from pathlib import Path

# Generacion de datos

## Demanda por hora y estacion

In [44]:
dataFile = '../data/interim/mibici_data.parquet'

df = pd.read_parquet(dataFile)
df

,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,2026-06-01 00:06:46,2026-06-01 00:09:53,16,30
1,2026-06-01 00:07:19,2026-06-01 00:26:47,32,175
2,2026-06-01 00:10:51,2026-06-01 00:31:56,263,270
3,2026-06-01 00:15:44,2026-06-01 00:21:47,38,81
4,2026-06-01 00:21:55,2026-06-01 00:35:08,26,159
...,...,...,...,...
7891822,2025-05-31 23:59:12,2025-06-01 00:06:32,79,69
7891823,2025-05-31 23:59:20,2025-06-01 00:17:54,70,34
7891824,2025-05-31 23:59:41,2025-06-01 00:13:55,143,233
7891825,2025-05-31 23:59:42,2025-06-01 00:10:31,75,28


Creación de dataframe "demanda" donde solo se considera la cantidad de bicicletas utilizadas (demanda) por unidad de tiempo (hora)

In [45]:
demanda = (
    df.groupby([
        "Origen_Id",
        pd.Grouper(key="Inicio_del_viaje", freq="h")
    ])
    .size()
    .reset_index(name="demanda")
)

demanda = demanda.rename(columns={"Inicio_del_viaje": "fecha_hora"})
demanda = demanda.sort_values(["Origen_Id", "fecha_hora"])
demanda

,Origen_Id,fecha_hora,demanda
0,10,2025-01-01 09:00:00,1
1,10,2025-01-01 10:00:00,2
2,10,2025-01-01 11:00:00,1
3,10,2025-01-01 13:00:00,2
4,10,2025-01-01 15:00:00,2
...,...,...,...
2696966,98,2026-09-30 18:00:00,2
2696967,98,2026-09-30 19:00:00,3
2696968,98,2026-09-30 20:00:00,1
2696969,98,2026-09-30 22:00:00,2


Con lo anterior se pueden crear situaciones donde no hay valores por ciertas horas lo que creara vacios, sin embargo se tienen que seguir considerando para que el modelo ML no sobre estime estos casos.

In [46]:
estaciones = demanda["Origen_Id"].unique()

rango_fechas = pd.date_range(
    start=demanda["fecha_hora"].min().floor("h"),
    end=demanda["fecha_hora"].max().floor("h"),
    freq="h"
)

indice_completo = pd.MultiIndex.from_product(
    [estaciones, rango_fechas],
    names=["Origen_Id", "fecha_hora"]
)

demanda = (
    demanda.set_index(["Origen_Id", "fecha_hora"])
    .reindex(indice_completo, fill_value=0) # A los campos vacios entre filas se rellena con valor 0 en demanda
    .reset_index()
)

Se crean los valores historicos para mostrar el comportamiento 1 hora antes, 1 dia antes y una semana antes.

In [47]:
demanda = demanda.sort_values(["Origen_Id", "fecha_hora"])

demanda["lag_1h"] = demanda.groupby("Origen_Id")["demanda"].shift(1)
demanda["lag_24h"] = demanda.groupby("Origen_Id")["demanda"].shift(24)
demanda["lag_7d"] = demanda.groupby("Origen_Id")["demanda"].shift(24 * 7)

demanda["media_24h"] = (
    demanda.groupby("Origen_Id")["demanda"]
    .transform(lambda x: x.shift(1).rolling(24).mean())
)

In [ ]:
demanda_modelo = demanda.dropna().copy()

# train = demanda_modelo[demanda_modelo["fecha_hora"] < "2026-06-01"]
# validacion = demanda_modelo[
#     (demanda_modelo["fecha_hora"] >= "2026-06-01") &
#     (demanda_modelo["fecha_hora"] < "2026-08-01")
# ]
# test = demanda_modelo[
#     demanda_modelo["fecha_hora"] >= "2026-08-01"
# ]

In [50]:
TRAIN_SPLIT = 0.80   # 80% train, 20% test

# Ordenar por fecha
demanda_modelo = demanda_modelo.sort_values("fecha_hora").copy()

# Fechas únicas ordenadas: así no separas una misma hora entre train y test
fechas_unicas = demanda_modelo["fecha_hora"].drop_duplicates().sort_values()

# Posición de corte
indice_corte = int(len(fechas_unicas) * TRAIN_SPLIT)

# Fecha a partir de la cual inicia test
fecha_corte = fechas_unicas.iloc[indice_corte]

# Split temporal
train = demanda_modelo[
    demanda_modelo["fecha_hora"] < fecha_corte
].copy()

test = demanda_modelo[
    demanda_modelo["fecha_hora"] >= fecha_corte
].copy()

print("Fecha de corte:", fecha_corte)
print(f"Train: {train['fecha_hora'].min()} a {train['fecha_hora'].max()}")
print(f"Test:  {test['fecha_hora'].min()} a {test['fecha_hora'].max()}")

print(f"% train: {len(train) / len(demanda_modelo):.2%}")
print(f"% test:  {len(test) / len(demanda_modelo):.2%}")

Fecha de corte: 2026-05-27 19:00:00
Train: 2025-01-08 00:00:00 a 2026-05-27 18:00:00
Test:  2026-05-27 19:00:00 a 2026-09-30 23:00:00
% train: 80.00%
% test:  20.00%


In [51]:
print("Train:", train["fecha_hora"].min(), "-", train["fecha_hora"].max())
print("Test:", test["fecha_hora"].min(), "-", test["fecha_hora"].max())

Train: 2025-01-08 00:00:00 - 2026-05-27 18:00:00
Test: 2026-05-27 19:00:00 - 2026-09-30 23:00:00


In [52]:
for data in [train, test]:
    data["hora"] = data["fecha_hora"].dt.hour
    data["dia_semana"] = data["fecha_hora"].dt.dayofweek  # lunes=0
    data["mes"] = data["fecha_hora"].dt.month
    data["es_fin_semana"] = (data["dia_semana"] >= 5).astype(int)

/tmp/ipykernel_219518/2208160952.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  data["dia_semana"] = data["fecha_hora"].dt.dayofweek  # lunes=0
/tmp/ipykernel_219518/2208160952.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  data["dia_semana"] = data["fecha_hora"].dt.dayofweek  # lunes=0
